#Dataset

# Overview

1. Read the raw data (e.g., u.data) from github
2. Sort and extract the data (e.g., 100-by-100 matrix) 
  * Use "create_data" function
3. Create the k-fold dataset (e.g., k=10) 
  * Use "k_fold_data" function
4. Save the k data files 

In [9]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [11]:
# Create n-by-m matrix (input: df, output: new_df)
def create_data(df, n, m):
  
  pivot_df = df.pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
  
  n_row = pivot_df.shape[0]
  n_col = pivot_df.shape[1]
  Obs_ind = np.where(pivot_df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  print(f'(Original) matrix sparsity: {sparsity:f}')

  #Sorting and extracting 
  new_df1 = pivot_df.copy()
  new_df1['count'] = new_df1.count(axis=1)
  new_df1_sort = new_df1.sort_values('count', ascending=False)
  new_df2 = new_df1_sort.drop("count", axis=1, inplace=False)
  new_df3 = new_df2.transpose()
  new_df3['count'] = new_df3.count(axis=1)
  new_df3_sort = new_df3.sort_values('count', ascending=False)  
  final_df = new_df3_sort.drop("count", axis=1, inplace=False)
  final_data = final_df.transpose()
  new_df = final_data.iloc[0:n,0:m]
  #print(new_df)
  
  n_row = new_df.shape[0]
  n_col = new_df.shape[1]
  Obs_ind = np.where(new_df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  sparsity = 1 - num_Obs / (m * n)

  print(f'(Extracted) matrix sparsity: {sparsity:f}')


  return new_df

In [91]:
#Create the k-fold datasets 
import random
def k_fold_data(df,k,random_seed):

  n_row = df.shape[0]
  n_col = df.shape[1]
  Obs_ind = np.where(df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  print(f'(Original) matrix sparsity: {sparsity:f}')

  a = int(num_Obs/k)
  A = k*a + k - num_Obs
  num_Obs_each = np.append(a*np.ones(A), (a+1)*np.ones(k-A)).astype(int)
  #print(num_Obs_each)

  label = np.array(range(num_Obs))

  p = [None] * k
  test_dat = [None] * k

  for i in range(k):

    Tdat = pd.DataFrame.copy(df)
    random.seed(random_seed)
    random.shuffle(label)
    p[i] = label[:num_Obs_each[i]]
    #print(p[i])
    for j in p[i]:
      Tdat.iloc[Obs_ind[0][j], Obs_ind[1][j]] = np.nan
    
    test_dat[i] = Tdat
    #print(test_dat[i])
    #print(len(np.where(test_dat[i].notnull())[0]))
    sparsity = 1 -  len(np.where(test_dat[i].notnull())[0]) / (test_dat[i].shape[0] * test_dat[i].shape[1])
    #print(str(i)+"th data sparsity = "+str(sparsity))

  return test_dat


In [211]:
# Set parameters
n_items = 100   
n_users = 700
k = 10 

In [212]:
#create the datasets 
random_seed = 20230911
# from google.colab import files
url = "https://raw.githubusercontent.com/park61/imputation/main/u.data"

colnames = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(url,sep='\t', names=colnames, header=None)
print(df.shape)
new_df = create_data(df, n_items, n_users)
new_df.to_csv('./data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv')

test_dat = k_fold_data(new_df,k,random_seed)

#Checking invalid dataset 
for i in range(k):
  print(i, min((1-test_dat[i].isna()).sum(axis=0)), min((1-test_dat[i].isna()).sum(axis=1)))

(100000, 4)
(Original) matrix sparsity: 0.936953
(Extracted) matrix sparsity: 0.609329
(Original) matrix sparsity: 0.609329
0 2 157
1 2 147
2 1 151
3 2 154
4 1 145
5 1 152
6 2 149
7 2 146
8 2 147
9 2 147


# 4. Save data

In [194]:
for i in range(k):
  if i<9:
      filename = './data/'+str(20230808)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(i+1)+'.csv'
  else:
      filename = './data/'+str(20230808)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(i+1)+'.csv'
  test_dat[i].to_csv(filename)